### Import packages

In [ ]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition

### Create model

In [ ]:
# Define model
m = ConcreteModel()

# Define sets
m.ITEMS = Set(initialize=['A', 'B', 'C'])

# Define Decision Variable
m.MarkPct = Var(m.ITEMS, domain = NonNegativeReals, bounds = (0,1))

# Define Implicit Variables
m.MarkPrice = Var(m.ITEMS, domain = NonNegativeReals)
m.Demand = Var(m.ITEMS, domain = NonNegativeReals)
m.Revenue = Var(m.ITEMS, domain = NonNegativeReals)
m.FinalInv = Var(m.ITEMS, domain = NonNegativeReals)

# Define Params
m.basePrice = Param(m.ITEMS, initialize = {'A':21.5, 'B':29.5, 'C': 30.5})
m.initInv = Param(m.ITEMS, initialize = {'A':10, 'B':30, 'C':41})
m.maxPct = 0.80
m.cap = 20
m.dispCost = 1.2

# Define equations for implicit variables

# Define Markdown Price
def Define_MarkPrice_Rule(m,i):
    return (1-m.MarkPct[i])*m.basePrice[i] == m.MarkPrice[i]
m.Define_MarkPrice =  Constraint(m.ITEMS, rule = Define_MarkPrice_Rule)

# Define Demand
def Define_Demand_Rule(m,i):
    return  m.MarkPct[i]*m.initInv[i] == m.Demand[i]
m.Define_Demand =  Constraint(m.ITEMS, rule = Define_Demand_Rule)

# Define Revenue
def Define_Revenue_Rule(m,i):
    return m.MarkPrice[i]*m.Demand[i] == m.Revenue[i]
m.Define_Revenue =  Constraint(m.ITEMS, rule = Define_Revenue_Rule)

# Define Final Inv
def Define_FinalInv_Rule(m,i):
    return  m.initInv[i] - m.Demand[i] == m.FinalInv[i]
m.Define_FinalInv =  Constraint(m.ITEMS, rule = Define_FinalInv_Rule)


# Define Constraints

# Define max discount
def Max_Discount_Rule(m,i):
    return m.MarkPct[i] <= m.maxPct
m.Max_Discount = Constraint(m.ITEMS, rule = Max_Discount_Rule)

# Sell such that total leftover inventory (all items) is at most cap
# Per-item alternative: m.FinalInv[i] <= m.cap, indexed by m.ITEMS
def Leftover_Inv_Rule(m):
    return sum(m.FinalInv[i] for i in m.ITEMS) <= m.cap
m.Leftover_Inv = Constraint(rule = Leftover_Inv_Rule)

# Define Objective Function
def Max_Revenue_Rule(m):
    return sum((m.Revenue[i] - m.dispCost*m.FinalInv[i]) for i in m.ITEMS)
m.Max_Revenue = Objective(rule = Max_Revenue_Rule, sense = maximize)

### Solve

In [ ]:
solver = SolverFactory('ipopt')
solution = solver.solve(m)

Check the status of the solver

In [ ]:
print(solution['Solver'])

### Extract Optimum Values

Print optimum value of objective function

In [ ]:
print(m.Max_Revenue.expr())

Print optimum value of decision and implicit variables - Option 1 (tailored syntax, need to substitute the right indexes and variable names for other use cases)

In [ ]:
data = {'ITEM': [], 'Opt_Discount': [], 'Opt_Price': [], 'Opt_Revenue': [], 'Final_Inv': []}
for i in m.ITEMS:
    data['ITEM'].append(i)
    data['Opt_Discount'].append(m.MarkPct[i].value)
    data['Opt_Price'].append(m.MarkPrice[i].value)
    data['Opt_Revenue'].append(m.Revenue[i].value)
    data['Final_Inv'].append(m.FinalInv[i].value)
pd.DataFrame(data)

Print optimum value of decision and implicit variables - Option 2 (generic, same syntax will work for other use cases)

In [ ]:
data = {'Variable': [], 'Value': []}
for var in m.component_objects(Var):
    for index in var:
        data['Variable'].append(f'{var.name}[{index}]')
        data['Value'].append(var[index].value)
pd.DataFrame(data)